# Aim 1: Prevalence of Psychiatric Disorders in Epilepsy Surgery

**Project:** Prevalence and Trends of DSM-5 Axis I Psychiatric Disorders in Epilepsy Surgery  
**Data:** Analytic cohort from `01_cohort_building.ipynb`

This notebook:
1. Computes **survey-weighted prevalence** of each psychiatric disorder
2. Compares prevalence across groups: surgery vs non-surgical epilepsy
3. Compares across surgical subgroups: resective vs VNS vs RNS vs monitoring
4. Creates Table 1 (demographics) and Table 2 (psychiatric prevalence)
5. Generates forest plot of prevalence ratios

In [ ]:
# ============================================================
# SETUP
# ============================================================
library(arrow)
library(data.table)
library(survey)
library(tableone)
library(ggplot2)
library(scales)
library(gridExtra)

outdir <- "/Volumes/Niels 2/NIS_new_version/NIS_epilepsy_psych"

cat("Loading analytic cohort...\n")
dt <- as.data.table(read_parquet(file.path(outdir, "output/epilepsy_analytic.parquet")))
cat(sprintf("Loaded: %s rows x %d columns\n", format(nrow(dt), big.mark = ","), ncol(dt)))
invisible(NULL)

In [ ]:
# ============================================================
# PREPARE VARIABLES
# ============================================================

# Create comparison group variable
dt[, group := fifelse(any_surgery, "Surgery", "Non-Surgical")]
dt[, group := factor(group, levels = c("Surgery", "Non-Surgical"))]

# Convert psychiatric flags to integer for survey functions
psych_cols <- grep("^psych_", names(dt), value = TRUE)
for (col in c(psych_cols, "any_psych")) {
    dt[[col]] <- as.integer(dt[[col]])
}

# Demographics as factors
dt[, female_f := factor(FEMALE, levels = c(0, 1), labels = c("Male", "Female"))]
dt[, race_f := factor(RACE, levels = 1:6,
                      labels = c("White", "Black", "Hispanic", "Asian/PI", "Native Am", "Other"))]
dt[, pay_f := factor(PAY1, levels = 1:6,
                     labels = c("Medicare", "Medicaid", "Private", "Self-pay", "No charge", "Other"))]
dt[, income_f := factor(ZIPINC_QRTL, levels = 1:4,
                        labels = c("Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"))]
dt[, intractable_f := factor(as.integer(intractable), levels = c(0, 1),
                             labels = c("Not intractable", "Intractable"))]

# Age groups
dt[, age_group := cut(AGE, breaks = c(18, 30, 45, 60, 75, Inf),
                      labels = c("18-29", "30-44", "45-59", "60-74", "75+"),
                      right = FALSE)]

cat("Variables prepared.\n")
cat(sprintf("Surgery: %s | Non-Surgical: %s\n",
            format(sum(dt$group == "Surgery"), big.mark = ","),
            format(sum(dt$group == "Non-Surgical"), big.mark = ",")))
invisible(NULL)

In [ ]:
# ============================================================
# CREATE SURVEY DESIGN
# ============================================================
options(survey.lonely.psu = "adjust")

svy <- svydesign(
    id      = ~HOSP_NIS,
    strata  = ~NIS_STRATUM,
    weights = ~DISCWT,
    nest    = TRUE,
    data    = dt
)

cat(sprintf("Survey design created: %s observations\n", format(nrow(dt), big.mark = ",")))
cat(sprintf("Weighted N: %s\n", format(round(sum(weights(svy))), big.mark = ",")))
invisible(NULL)

## Table 1: Demographics by Surgery Status

In [ ]:
# ============================================================
# TABLE 1: Demographics by surgery status (unweighted for Table 1)
# ============================================================
tab1_vars <- c("AGE", "female_f", "race_f", "pay_f", "income_f",
               "epilepsy_type", "intractable_f",
               "cm_hypertension", "cm_diabetes", "cm_obesity",
               "cm_cvd", "cm_brain_tumor",
               "DIED", "LOS")

tab1_factor <- c("female_f", "race_f", "pay_f", "income_f",
                 "epilepsy_type", "intractable_f",
                 "cm_hypertension", "cm_diabetes", "cm_obesity",
                 "cm_cvd", "cm_brain_tumor", "DIED")

# Convert comorbidity/outcome flags to factor for tableone
for (v in c("cm_hypertension", "cm_diabetes", "cm_obesity", "cm_cvd",
            "cm_brain_tumor", "DIED")) {
    dt[[v]] <- factor(as.integer(dt[[v]]), levels = c(0, 1), labels = c("No", "Yes"))
}

tab1 <- CreateTableOne(
    vars       = tab1_vars,
    strata     = "group",
    data       = dt,
    factorVars = tab1_factor,
    test       = TRUE
)

print(tab1, showAllLevels = TRUE, formatOptions = list(big.mark = ","))
invisible(NULL)

In [ ]:
# ============================================================
# Save Table 1 as CSV
# ============================================================
tab1_csv <- print(tab1, showAllLevels = TRUE, quote = FALSE, noSpaces = TRUE,
                  printToggle = FALSE, formatOptions = list(big.mark = ","))
write.csv(tab1_csv, file.path(outdir, "tables/table1_demographics.csv"))
cat("Saved tables/table1_demographics.csv\n")
invisible(NULL)

## Table 2: Survey-Weighted Psychiatric Prevalence by Group

In [ ]:
# ============================================================
# TABLE 2: Survey-weighted psychiatric prevalence
# ============================================================
psych_labels <- c(
    "psych_depression"     = "Depression (MDD)",
    "psych_bipolar"        = "Bipolar disorder",
    "psych_anxiety"        = "Anxiety disorders",
    "psych_ptsd"           = "PTSD",
    "psych_ocd"            = "OCD",
    "psych_schizophrenia"  = "Schizophrenia spectrum",
    "psych_psychosis"      = "Other psychosis",
    "psych_adhd"           = "ADHD",
    "psych_alcohol"        = "Alcohol use disorder",
    "psych_drug"           = "Drug use disorder",
    "psych_suicidal"       = "Suicidal ideation",
    "psych_pnes"           = "PNES (conversion)",
    "psych_organic"        = "Organic psychiatric",
    "any_psych"            = "ANY PSYCHIATRIC"
)

# Survey subsets
svy_surg    <- subset(svy, group == "Surgery")
svy_nonsurg <- subset(svy, group == "Non-Surgical")

results <- data.frame(
    Disorder = character(),
    Prev_Surgery = character(),
    Prev_NonSurg = character(),
    Prev_Overall = character(),
    P_value = numeric(),
    stringsAsFactors = FALSE
)

cat(sprintf("%-25s  %15s  %15s  %15s  %10s\n",
            "Disorder", "Surgery %", "Non-Surg %", "Overall %", "p-value"))
cat(paste(rep("-", 85), collapse = ""), "\n")

for (col in names(psych_labels)) {
    fmla <- as.formula(paste0("~", col))

    # Overall
    m_all <- svymean(fmla, svy, na.rm = TRUE)
    ci_all <- confint(m_all)
    prev_all <- sprintf("%.1f (%.1f-%.1f)", 100 * coef(m_all)[1],
                        100 * ci_all[1, 1], 100 * ci_all[1, 2])

    # Surgery
    m_s <- svymean(fmla, svy_surg, na.rm = TRUE)
    ci_s <- confint(m_s)
    prev_s <- sprintf("%.1f (%.1f-%.1f)", 100 * coef(m_s)[1],
                      100 * ci_s[1, 1], 100 * ci_s[1, 2])

    # Non-surgical
    m_ns <- svymean(fmla, svy_nonsurg, na.rm = TRUE)
    ci_ns <- confint(m_ns)
    prev_ns <- sprintf("%.1f (%.1f-%.1f)", 100 * coef(m_ns)[1],
                       100 * ci_ns[1, 1], 100 * ci_ns[1, 2])

    # Chi-square test
    p <- tryCatch({
        test <- svychisq(as.formula(paste0("~", col, " + group")), svy)
        test$p.value
    }, error = function(e) NA)

    p_str <- ifelse(is.na(p), "N/A",
                    ifelse(p < 0.001, "<0.001", sprintf("%.3f", p)))

    cat(sprintf("%-25s  %15s  %15s  %15s  %10s\n",
                psych_labels[col], prev_s, prev_ns, prev_all, p_str))

    results <- rbind(results, data.frame(
        Disorder = psych_labels[col],
        Prev_Surgery = prev_s,
        Prev_NonSurg = prev_ns,
        Prev_Overall = prev_all,
        P_value = p,
        stringsAsFactors = FALSE
    ))
}

write.csv(results, file.path(outdir, "tables/table2_psych_prevalence.csv"), row.names = FALSE)
cat("\nSaved tables/table2_psych_prevalence.csv\n")
invisible(NULL)

## Table 3: Psychiatric Prevalence by Surgical Subgroup

In [ ]:
# ============================================================
# TABLE 3: Prevalence by surgical subgroup
# ============================================================
subgroups <- c("Resective", "VNS", "RNS")

sub_results <- list()
for (sg in subgroups) {
    svy_sg <- subset(svy, surgery_group == sg)
    sg_res <- data.frame(Subgroup = sg, stringsAsFactors = FALSE)
    n_sg <- nrow(dt[surgery_group == sg])
    sg_res$N <- n_sg

    for (col in names(psych_labels)) {
        fmla <- as.formula(paste0("~", col))
        m <- tryCatch({
            svymean(fmla, svy_sg, na.rm = TRUE)
        }, error = function(e) NULL)

        if (!is.null(m)) {
            ci <- confint(m)
            sg_res[[psych_labels[col]]] <- sprintf("%.1f (%.1f-%.1f)",
                100 * coef(m)[1], 100 * ci[1, 1], 100 * ci[1, 2])
        } else {
            sg_res[[psych_labels[col]]] <- "N/A"
        }
    }
    sub_results[[sg]] <- sg_res
}

sub_df <- do.call(rbind, sub_results)
rownames(sub_df) <- NULL

# Print nicely
cat("Psychiatric prevalence by surgical subgroup (survey-weighted %, 95% CI):\n\n")
for (i in 1:nrow(sub_df)) {
    cat(sprintf("\n--- %s (N=%s) ---\n", sub_df$Subgroup[i], format(sub_df$N[i], big.mark = ",")))
    for (col in names(psych_labels)) {
        cat(sprintf("  %-25s %s\n", psych_labels[col], sub_df[[psych_labels[col]]][i]))
    }
}

write.csv(sub_df, file.path(outdir, "tables/table3_psych_by_subgroup.csv"), row.names = FALSE)
cat("\nSaved tables/table3_psych_by_subgroup.csv\n")
invisible(NULL)

## Figure 1: Forest Plot of Psychiatric Prevalence (Surgery vs Non-Surgical)

In [ ]:
# ============================================================
# FIGURE 1: Forest plot — prevalence comparison
# ============================================================
# Build plot data
plot_data <- data.frame(
    Disorder = character(),
    Group = character(),
    Prev = numeric(),
    Lower = numeric(),
    Upper = numeric(),
    stringsAsFactors = FALSE
)

for (col in names(psych_labels)) {
    if (col == "any_psych") next  # plot separately
    fmla <- as.formula(paste0("~", col))

    for (grp in c("Surgery", "Non-Surgical")) {
        svy_sub <- if (grp == "Surgery") svy_surg else svy_nonsurg
        m <- svymean(fmla, svy_sub, na.rm = TRUE)
        ci <- confint(m)
        plot_data <- rbind(plot_data, data.frame(
            Disorder = psych_labels[col],
            Group = grp,
            Prev = 100 * coef(m)[1],
            Lower = 100 * ci[1, 1],
            Upper = 100 * ci[1, 2],
            stringsAsFactors = FALSE
        ))
    }
}

# Order disorders by overall prevalence (descending)
disorder_order <- plot_data[plot_data$Group == "Non-Surgical", ]
disorder_order <- disorder_order[order(disorder_order$Prev), "Disorder"]
plot_data$Disorder <- factor(plot_data$Disorder, levels = disorder_order)

p1 <- ggplot(plot_data, aes(x = Prev, y = Disorder, color = Group)) +
    geom_point(position = position_dodge(width = 0.6), size = 2.5) +
    geom_errorbarh(aes(xmin = Lower, xmax = Upper),
                   position = position_dodge(width = 0.6), height = 0.3) +
    scale_color_manual(values = c("Surgery" = "#E63946", "Non-Surgical" = "#457B9D")) +
    labs(x = "Survey-Weighted Prevalence (%)",
         y = NULL,
         title = "Prevalence of Psychiatric Disorders in Epilepsy",
         subtitle = "Surgery vs Non-Surgical (NIS 2011-2020, survey-weighted, 95% CI)",
         color = "Group") +
    theme_minimal(base_size = 12) +
    theme(legend.position = "bottom",
          panel.grid.minor = element_blank())

print(p1)

ggsave(file.path(outdir, "figures/fig1_prevalence_forest.pdf"), p1,
       width = 10, height = 7, dpi = 300)
ggsave(file.path(outdir, "figures/fig1_prevalence_forest.png"), p1,
       width = 10, height = 7, dpi = 300)
cat("Saved figures/fig1_prevalence_forest.pdf/.png\n")
invisible(NULL)

## Figure 2: Grouped Bar Chart by Surgical Subgroup

In [ ]:
# ============================================================
# FIGURE 2: Bar chart — key disorders by surgical subgroup
# ============================================================
key_disorders <- c("psych_depression", "psych_anxiety", "psych_bipolar",
                   "psych_alcohol", "psych_drug", "psych_adhd", "any_psych")

bar_data <- data.frame()
for (sg in c("Resective", "VNS", "RNS", "Non-Surgical")) {
    if (sg == "Non-Surgical") {
        svy_sub <- svy_nonsurg
    } else {
        svy_sub <- subset(svy, surgery_group == sg)
    }

    for (col in key_disorders) {
        m <- tryCatch({
            svymean(as.formula(paste0("~", col)), svy_sub, na.rm = TRUE)
        }, error = function(e) NULL)

        if (!is.null(m)) {
            ci <- confint(m)
            bar_data <- rbind(bar_data, data.frame(
                Subgroup = sg,
                Disorder = psych_labels[col],
                Prev = 100 * coef(m)[1],
                Lower = 100 * ci[1, 1],
                Upper = 100 * ci[1, 2]
            ))
        }
    }
}

bar_data$Subgroup <- factor(bar_data$Subgroup,
                            levels = c("Resective", "VNS", "RNS", "Non-Surgical"))

p2 <- ggplot(bar_data, aes(x = Disorder, y = Prev, fill = Subgroup)) +
    geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.7) +
    geom_errorbar(aes(ymin = pmax(Lower, 0), ymax = Upper),
                  position = position_dodge(width = 0.8), width = 0.3) +
    scale_fill_brewer(palette = "Set2") +
    labs(x = NULL, y = "Survey-Weighted Prevalence (%)",
         title = "Psychiatric Comorbidity by Epilepsy Surgery Type",
         subtitle = "NIS 2012-2020, survey-weighted with 95% CI",
         fill = "Group") +
    theme_minimal(base_size = 11) +
    theme(axis.text.x = element_text(angle = 30, hjust = 1),
          legend.position = "bottom",
          panel.grid.minor = element_blank())

print(p2)

ggsave(file.path(outdir, "figures/fig2_subgroup_bars.pdf"), p2,
       width = 12, height = 7, dpi = 300)
ggsave(file.path(outdir, "figures/fig2_subgroup_bars.png"), p2,
       width = 12, height = 7, dpi = 300)
cat("Saved figures/fig2_subgroup_bars.pdf/.png\n")
invisible(NULL)

## Multivariable Logistic Regression: Predictors of Any Psychiatric Comorbidity

In [ ]:
# ============================================================
# MULTIVARIABLE LOGISTIC: Predictors of any psychiatric comorbidity
# ============================================================
# Survey-weighted logistic regression
model_psych <- svyglm(
    any_psych ~ group + AGE + female_f + race_f + pay_f + income_f +
        epilepsy_type + intractable_f + icd_version,
    design = svy,
    family = quasibinomial()
)

cat("Survey-weighted logistic regression: Predictors of any psychiatric comorbidity\n\n")
summary_model <- summary(model_psych)
print(summary_model)

# ORs with CI
coefs <- coef(model_psych)
ci <- confint(model_psych)
or_table <- data.frame(
    Variable = names(coefs),
    OR = exp(coefs),
    Lower = exp(ci[, 1]),
    Upper = exp(ci[, 2]),
    P = summary_model$coefficients[, "Pr(>|t|)"]
)
rownames(or_table) <- NULL

cat("\n\nOdds Ratios (95% CI):\n")
for (i in 1:nrow(or_table)) {
    p_str <- ifelse(or_table$P[i] < 0.001, "<0.001", sprintf("%.3f", or_table$P[i]))
    cat(sprintf("  %-35s OR %.2f (%.2f-%.2f) p=%s\n",
                or_table$Variable[i], or_table$OR[i],
                or_table$Lower[i], or_table$Upper[i], p_str))
}

write.csv(or_table, file.path(outdir, "tables/logistic_any_psych.csv"), row.names = FALSE)
cat("\nSaved tables/logistic_any_psych.csv\n")
invisible(NULL)